# 11 · Leave-one-out: does `fresh` really beat `finetune`?

Notebook 10 V1 found `fresh` (random init) beating `finetune` (from `winner_aug`) on M0 and
M2, contradicting the prediction made before the run. But that was **one holdout cube, one
seed**, and this project has twice measured how unreliable that is: V7/V9 saw M2 swing
+18.4% → +2.5% on the same cube with zero config change, and v25 saw 1.73 dB from an
identical-seed rerun. A 10–15pp gap on n=1 sits inside that.

Five disks means leave-one-out is available: **5 folds, each 3 train / 1 val / 1 holdout,
every cube taking a turn as holdout.** That converts one number into a spread across cubes,
which is what RULES.md #6 asks for.

Training seed is held fixed at 42 across all folds, so the only thing varying is which cube is
held out. This isolates *cube* variance; training variance would need seed repeats on top and
is deliberately not measured here.

**Expect one strange fold.** `run_9032` is an odd cube: its synthesized pair came out at
rmsdiff 0.107 against the others' 0.46–0.54, and its signal mask covers 98.9% of the field.
When it is the holdout, that fold's numbers will not look like the rest. Recorded here in
advance so it is not explained away afterwards.

**Runtime ~3 h** (5 folds × 2 arms × ~18 min). Results are written after every fold, so an
interrupted session still yields usable partial data.

## 0. Bootstrap

In [1]:
import os, sys, subprocess, glob

ON_KAGGLE = os.path.exists('/kaggle')
BRANCH = 'midterm-prep'
if ON_KAGGLE:
    REPO = '/kaggle/working/EXXA'; PKG = os.path.join(REPO, 'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',
                        'https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/'+BRANCH], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps',
                    'pytorch-msssim','bettermoments'], check=True)
    os.chdir(os.path.join(PKG,'notebooks')); sys.path.insert(0, PKG)

    hits = [p for p in glob.glob('/kaggle/input/**/run_9*_rt_*', recursive=True) if os.path.isdir(p)]
    if not hits:
        raise FileNotFoundError('No run_9*_rt_* folders under /kaggle/input -- attach exxa-sg-synth-pairs.')
    DATA_DIR = os.path.dirname(hits[0])
    ck = glob.glob('/kaggle/input/**/winner_aug_seed43.*', recursive=True)
    if not ck:
        raise FileNotFoundError('winner_aug_seed43 checkpoint not found under /kaggle/input.')
    WINNER_CKPT = ck[0]
else:
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'):
        os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
    DATA_DIR = '../self-gravitating cube and dirty cube/sg_synth'
    WINNER_CKPT = '../models/08-seeds/winner_aug_seed43.pth'

print('DATA_DIR:   ', DATA_DIR)
print('WINNER_CKPT:', WINNER_CKPT)

Cloning into '/kaggle/working/EXXA'...
Updating files: 100% (5440/5440), done.


DATA_DIR:    /kaggle/input/datasets/krishanyadav333/exxa-sg-synth-pairs/kaggle-sg-training-dataset
WINNER_CKPT: /kaggle/input/datasets/krishanyadav333/exxa-sg-synth-pairs/kaggle-sg-training-dataset/winner_aug_seed43.ckpt


## 0b. Pull latest `src/` (RULES.md #2: this updates the library, never these cells)

In [2]:
if ON_KAGGLE:
    subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','FETCH_HEAD'], check=True)
    print(subprocess.run(['git','-C',REPO,'log','--oneline','-1'], capture_output=True, text=True).stdout)
    import importlib, src; importlib.reload(src)

From https://github.com/KrishanYadav333/EXXA
 * branch            midterm-prep -> FETCH_HEAD


HEAD is now at b7b140d feat: notebook 11, leave-one-out over the five SG disks
b7b140d feat: notebook 11, leave-one-out over the five SG disks



## 1. Config

In [3]:
import time, json, math
import numpy as np
import torch
import torch.nn.functional as Fn
from torch.utils.data import DataLoader
from astropy.io import fits

from src.data.cube_split import list_cubes
from src.data.fits_cube_dataset import FITSChannelDataset
from src.training.sweep import train_unet, val_metrics
from src.training.architectures import build_model
from src.models.unet import UNet
from src.evaluation.moment_maps import generate_moment_maps, moment_improvement

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42                     # held fixed across folds: only the fold varies
TARGET_SIZE = 256
N_SAMPLES = 120
NW = 2 if torch.cuda.is_available() else 0

WINNER = dict(base_channels=48, channel_multipliers=(1, 2, 4, 8),
              lr=8.196504330730313e-4, alpha=0.8877681051398497,
              sched_patience=8, batch_size=8)
FINETUNE_LR_SCALE = 0.1       # full LR overwrites the pretrained weights in the first steps

CKPT_DIR = '../results/checkpoints'
OUT_JSON = '../results/self-gravitating/sg_loo_results.json'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs('../results/self-gravitating', exist_ok=True)
print(f'{device} | seed {SEED} fixed across folds')

cuda | seed 42 fixed across folds


## 2. Folds

Built explicitly rather than by reseeding `split_cubes`, so every cube is holdout exactly
once and the assignment is auditable rather than a function of a seed.

In [4]:
cubes = sorted(list_cubes(DATA_DIR), key=lambda c: c['folder'])
n = len(cubes)
assert n == 5, f'expected 5 cubes, found {n}'

FOLDS = []
for i in range(n):
    hold = cubes[i]
    val = cubes[(i + 1) % n]
    train = [c for j, c in enumerate(cubes) if j != i and j != (i + 1) % n]
    FOLDS.append(dict(fold=i, holdout=hold, val=val, train=train))

for f in FOLDS:
    print(f"fold {f['fold']}: holdout {f['holdout']['folder']:24s} "
          f"val {f['val']['folder']:24s} train {[c['run_id'] for c in f['train']]}")

# leakage check: no cube may appear in more than one role within a fold
for f in FOLDS:
    ids = [f['holdout']['run_id'], f['val']['run_id']] + [c['run_id'] for c in f['train']]
    assert len(set(ids)) == len(ids), f"fold {f['fold']}: a cube appears twice"
print('\nno cube appears in two roles in any fold')

fold 0: holdout run_9015_00370_rt_00     val run_9019_00019_rt_00     train ['9025', '9032', '9074']
fold 1: holdout run_9019_00019_rt_00     val run_9025_00370_rt_00     train ['9015', '9032', '9074']
fold 2: holdout run_9025_00370_rt_00     val run_9032_00020_rt_00     train ['9015', '9019', '9074']
fold 3: holdout run_9032_00020_rt_00     val run_9074_00025_rt_00     train ['9015', '9019', '9025']
fold 4: holdout run_9074_00025_rt_00     val run_9015_00370_rt_00     train ['9019', '9025', '9032']

no cube appears in two roles in any fold


## 3. Load `winner_aug` (the `frozen` arm, and `finetune`'s starting point)

In [5]:
_ck = torch.load(WINNER_CKPT, map_location='cpu', weights_only=False)
WINNER_SD = _ck['model_state_dict']
assert _ck['base_channels'] == WINNER['base_channels']
assert tuple(_ck['channel_multipliers']) == WINNER['channel_multipliers']
print(f"winner_aug: epoch {_ck['epoch']}, val_loss {_ck['val_loss']:.6f}")

winner_aug: epoch 46, val_loss 0.000914


## 4. Helpers

In [6]:
def make_net(sd=None):
    net = build_model('unet', base_channels=WINNER['base_channels'],
                      channel_multipliers=WINNER['channel_multipliers'],
                      use_beam=False, n_neighbors=0, out_channels=1, latent_dim=128).to(device)
    if sd is not None:
        miss, unexp = net.load_state_dict(sd, strict=False)
        assert not miss and not unexp, f'state dict mismatch: {len(miss)}/{len(unexp)}'
    return net


def denoise_cube(sd, cube):
    net = make_net(sd); net.eval()
    C, H, W = cube.shape
    los = cube.reshape(C, -1).min(axis=1); his = cube.reshape(C, -1).max(axis=1)
    rng = his - los
    out = np.empty_like(cube, dtype=np.float32)
    with torch.no_grad():
        for s in range(0, C, 8):
            blk = cube[s:s+8].astype(np.float64)
            lo, hi = los[s:s+8], his[s:s+8]
            den = np.where((hi - lo) > 0, hi - lo, 1)[:, None, None]
            t = torch.from_numpy((blk - lo[:, None, None]) / den)[:, None].float().to(device)
            t = Fn.interpolate(t, (TARGET_SIZE, TARGET_SIZE), mode='bilinear', align_corners=False)
            p = net(t, torch.zeros(t.size(0), dtype=torch.long, device=device), None)
            b = Fn.interpolate(p, (H, W), mode='bilinear', align_corners=False)[:, 0].cpu().numpy()
            for k in range(b.shape[0]):
                out[s+k] = b[k] * rng[s+k] + los[s+k] if rng[s+k] > 0 else los[s+k]
    return out


def score_holdout(sd, hold):
    with fits.open(hold['clean'], memmap=True) as h:
        hdr = h[0].header; clean = h[0].data[:]
    with fits.open(hold['dirty'], memmap=True) as h:
        dirty = h[0].data[:]
    velax = (hdr['CRVAL3'] + (np.arange(clean.shape[0]) + 1 - hdr['CRPIX3']) * hdr['CDELT3']) * 1000.0
    m_clean = generate_moment_maps('', data_velax=(clean.astype(np.float64), velax))
    m_dirty = generate_moment_maps('', data_velax=(dirty.astype(np.float64), velax))
    den = denoise_cube(sd, dirty)
    m_den = generate_moment_maps('', data_velax=(den.astype(np.float64), velax))
    return moment_improvement(m_clean, m_dirty, m_den)

## 5. Run the folds

Results are appended and written to disk after **every** fold, so a session that dies at fold
3 still leaves three usable folds rather than nothing (RULES.md #1, applied to results).

In [7]:
all_results = []
t_start = time.time()

for f in FOLDS:
    fi = f['fold']
    print(f"\n{'#'*70}\n# FOLD {fi}  holdout={f['holdout']['folder']}\n{'#'*70}")

    _kw = dict(n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED,
               subtract_continuum=False, verbose=False)
    train_ds = FITSChannelDataset(f['train'], **_kw)
    val_ds = FITSChannelDataset([f['val']], **_kw)
    val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)
    print(f"  train {len(train_ds)} / {len(f['train'])} cubes | val {len(val_ds)}")

    states = {'frozen': WINNER_SD}
    trained = {}
    for arm, init, scale in (('finetune', WINNER_SD, FINETUNE_LR_SCALE), ('fresh', None, 1.0)):
        cfg = dict(WINNER); cfg['lr'] = cfg['lr'] * scale
        t0 = time.time()
        out = train_unet(train_ds, val_ds, device, init_state_dict=init,
                         min_epochs=10, max_epochs=60, patience=8,
                         num_workers=NW, seed=SEED,
                         ckpt_path=os.path.join(CKPT_DIR, f'loo{fi}_{arm}.pth'),
                         verbose=False, **cfg)
        out['minutes'] = (time.time() - t0) / 60
        trained[arm] = out
        _c = torch.load(os.path.join(CKPT_DIR, f'loo{fi}_{arm}.pth'), map_location='cpu', weights_only=False)
        states[arm] = _c['model_state_dict']
        print(f"  {arm:9s} PSNR {out['psnr']:.3f} best_ep {out['best_epoch']:2d} "
              f"({out['minutes']:.1f} min)")

    row = dict(fold=fi, holdout=f['holdout']['folder'], val=f['val']['folder'],
               train=[c['folder'] for c in f['train']], arms={})
    for arm, sd in states.items():
        pix = val_metrics(make_net(sd), val_loader, device, use_beam=False, arch='unet')
        mom = score_holdout(sd, f['holdout'])
        row['arms'][arm] = dict(psnr=pix['psnr'], ssim=pix['ssim'], mse=pix['mse'],
                                M0=mom['M0'], M1=mom['M1'], M2=mom['M2'],
                                minutes=trained.get(arm, {}).get('minutes'))
        print(f"    {arm:9s} PSNR {pix['psnr']:7.3f}  M0 {mom['M0']:+7.1f}  "
              f"M1 {mom['M1']:+7.1f}  M2 {mom['M2']:+7.1f}")

    all_results.append(row)
    with open(OUT_JSON, 'w') as fh:                       # persist after EVERY fold
        json.dump(dict(seed=SEED, n_folds=len(FOLDS), folds=all_results), fh, indent=2)
    print(f"  [saved {len(all_results)}/{len(FOLDS)} folds, {(time.time()-t_start)/60:.0f} min elapsed]")


######################################################################
# FOLD 0  holdout=run_9015_00370_rt_00
######################################################################
  train 360 / 3 cubes | val 120
  finetune  PSNR 35.991 best_ep  1 (5.5 min)
  fresh     PSNR 27.687 best_ep 26 (18.7 min)
    frozen    PSNR  34.558  M0   +23.6  M1    -0.6  M2   -22.3
    finetune  PSNR  35.991  M0   +35.0  M1   +26.3  M2    +9.3
    fresh     PSNR  27.687  M0   +37.4  M1   +17.9  M2    +7.4
  [saved 1/5 folds, 25 min elapsed]

######################################################################
# FOLD 1  holdout=run_9019_00019_rt_00
######################################################################
  train 360 / 3 cubes | val 120
  finetune  PSNR 30.638 best_ep 24 (17.7 min)
  fresh     PSNR 30.281 best_ep 26 (19.0 min)
    frozen    PSNR  29.032  M0   +23.8  M1    +9.7  M2   -76.3
    finetune  PSNR  30.638  M0   -24.1  M1    -7.5  M2   -44.0
    fresh     PSNR  30.281  M0    -6.3

## 6. Aggregate — the spread across cubes

In [8]:
arms = ['frozen', 'finetune', 'fresh']
print('per fold:\n')
print(f"{'fold':>4} {'holdout':24s} " + ' '.join(f'{a:>22s}' for a in arms))
print(f"{'':>4} {'':24s} " + ' '.join(f"{'M0':>7}{'M1':>7}{'M2':>8}" for _ in arms))
for r in all_results:
    cells_ = ' '.join(f"{r['arms'][a]['M0']:7.1f}{r['arms'][a]['M1']:7.1f}{r['arms'][a]['M2']:8.1f}"
                      for a in arms)
    print(f"{r['fold']:>4} {r['holdout'][:24]:24s} {cells_}")

print('\nmean +/- std across folds (spread is ACROSS CUBES, not seeds -- RULES.md #6):\n')
print(f"{'arm':10s} {'PSNR':>16} {'M0':>16} {'M1':>16} {'M2':>16}")
agg = {}
for a in arms:
    vals = {k: np.array([r['arms'][a][k] for r in all_results], dtype=float)
            for k in ('psnr', 'M0', 'M1', 'M2')}
    agg[a] = {k: dict(mean=float(v.mean()), std=float(v.std(ddof=1)) if len(v) > 1 else 0.0)
              for k, v in vals.items()}
    print(f"{a:10s} " + ' '.join(f"{vals[k].mean():9.2f} +/-{vals[k].std(ddof=1):5.2f}"
                                 for k in ('psnr', 'M0', 'M1', 'M2')))

# The question this notebook exists to answer.
d = {k: np.array([r['arms']['fresh'][k] - r['arms']['finetune'][k] for r in all_results], dtype=float)
     for k in ('M0', 'M1', 'M2')}
print('\nfresh minus finetune, per moment:')
for k, v in d.items():
    wins = int((v > 0).sum())
    # Deliberately conservative: compared against the full spread, not the standard error.
    # n=5 supports no significance claim either way, so the wording must not imply one.
    verdict = ('smaller than the fold-to-fold spread, so not separable here'
               if abs(v.mean()) < v.std(ddof=1) else
               'larger than the fold-to-fold spread, worth a seed repeat to confirm')
    print(f"  {k}: {v.mean():+7.2f} +/- {v.std(ddof=1):5.2f} pp, fresh wins {wins}/{len(v)} folds")
    print(f"      {verdict}")

with open(OUT_JSON, 'w') as fh:
    json.dump(dict(seed=SEED, n_folds=len(FOLDS), folds=all_results, aggregate=agg), fh, indent=2)
print(f'\nsaved -> {OUT_JSON}')
print(f'checkpoints: {CKPT_DIR}/loo*_*.pth  ({2*len(FOLDS)} files, ~112 MB each)')

per fold:

fold holdout                                  frozen               finetune                  fresh
                                   M0     M1      M2      M0     M1      M2      M0     M1      M2
   0 run_9015_00370_rt_00        23.6   -0.6   -22.3    35.0   26.3     9.3    37.4   17.9     7.4
   1 run_9019_00019_rt_00        23.8    9.7   -76.3   -24.1   -7.5   -44.0    -6.3  -20.7  -139.7
   2 run_9025_00370_rt_00         4.6   12.3    -9.5    13.2   22.3     7.1    14.2   19.2     7.7
   3 run_9032_00020_rt_00      -111.3 -284.2  -112.8   -68.2  -92.8   -36.9  -288.7 -351.5  -246.5
   4 run_9074_00025_rt_00       -10.3   -0.6   -43.6   -15.1   20.3     6.2    10.7    2.2   -48.4

mean +/- std across folds (spread is ACROSS CUBES, not seeds -- RULES.md #6):

arm                    PSNR               M0               M1               M2
frozen         29.53 +/- 2.98    -13.93 +/-56.28    -52.70 +/-129.56    -52.91 +/-41.98
finetune       30.88 +/- 3.03    -11.84 +/-39.27 